# Are the other three "no-highway escapes" sweepers?

## Question
`RLLLLRRL`, `RLLLRLRLLL` and `RLLLLRLRRL` are the only rules of the 1e11 census that "escaped" without a highway. The parent experiment turned the column sweeper
of `RLRLLL` into a checked macro-transition. Do these three run the same kind of sweeper, so that the same checker can be generalised? "Escaped" only means the
ant touched the edge of a 32768 board, and the three final boxes are nearly square, so a board-filling growth is an equally good reading. The registration (README)
therefore treats sweeper, missed highway, filler and inconclusive as competing outcomes.

## Assumptions
Blank board, ant at the centre heading north, plain loop of the other tracers. Exact-clock criterion: `dt = a * swing + b[k mod p]` with zero residual on >= 95% of the
cycles of a window, cycles defined by the front advancing P columns in one of four directions (P <= 8), phase period p <= 6 (registered). Windows: the last 2e8 steps
before the census exit step, and the last 2e8 steps of the run. Instrument accepted only after an oracle (below). Everything here reads committed files in `results/`.

In [1]:
import hashlib, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-sweeper-generalisation/results"
for n in sorted(p.name for p in R.glob("*.txt")):
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip())

eb36dfb52524cede g5_compare.txt
26581084b9887119 oracle.txt
747b90957b1cfe27 spiral_probe_analysis.txt
155897079d542418 spiral_steady.txt
9b3958f9465fc3e4 stage1_analysis.txt
b90577aec3336d0a stage1_binary.txt
a7decdc1b19093b0 stage1_negctl_extra.txt
40911ca1b9c48b39 stage1_onset.txt
2266b21ac7f776af stage1_onset_context.txt
67648e29350edeff stage1_posthoc.txt
b34f4e656df5ca8d stage1_posthoc_maxp64.txt
86b96da7dba42abb stage1_posthoc_rlllrrl.txt
git cb727d1


## Instrument oracle
Fingerprint (position, heading, hash of every cell) of the front-log tool equals `fast::run` at 1e7 and 1e8 steps for five rules, and an independent Python ant reproduces all four event logs for 2e5 steps.

In [2]:
print("\n".join(l for l in (R / "oracle.txt").read_text().splitlines() if not l.startswith("   Compiling")))

    Finished `release` profile [optimized] target(s) in 14.38s
# commit 1461cb9 dirty=5
7d135c9bc49b5c919b012d85c11f063537919ad7f49bb3c259e64cfb4bcd3ab9  /home/tim/turmite-target-sg/release/examples/sweeper_generalisation_frontlog
4de9a375ad29e8f906b76586080fcfff133c43e99f60480a82030f13948acaac  /home/tim/turmite-target-sg/release/examples/rlrlll_macro_transition_resim
FINGERPRINT RLLLLRRL 10000000 identical (12d684dc1b39ba4f 2036562)
FINGERPRINT RLLLLRRL 100000000 identical (c75d9ef62d0f0fc1 24689433)
xmin: 76 rust events, 76 python events, identical=True
xmax: 77 rust events, 77 python events, identical=True
ymin: 76 rust events, 76 python events, identical=True
ymax: 79 rust events, 79 python events, identical=True
ORACLE_LOG PASS
FINGERPRINT RLLLLRLRRL 10000000 identical (b0b824cd59c8aaa7 304924)
FINGERPRINT RLLLLRLRRL 100000000 identical (ff34f0f251632ef3 1059431)
xmin: 42 rust events, 42 python events, identical=True
xmax: 38 rust events, 38 python events, identical=True
ymin: 38

## Registered Stage 1 analysis
Positive control `RLRLLL` should be found; the three targets are the subject. Lines are the best candidate per window, the fraction of cycles with an exactly zero residual, and the new-cell window signature S of the parent (steps per new cell below 8 and box aspect at least 20).

In [3]:
print((R / "stage1_analysis.txt").read_text())

# commit cd802ed8b014f0368826e2ec3cf50bf29423c8c1
=== ctl_pos_RLRLLL [12677100000, 12974495927] positive control, post-switch part
ymax P=8: cycles=293 distinct_swing=4 best p=5 a=-7652 exact=0.0341 b=[866924, 853100, 862316, 871532, 857708]
candidates tried (direction x P x p): 138
BEST EXACT_CLOCK: dir=xmin P=3 p=3 a=12 exact_fraction=1.0000 cycles=4714 b=[326, 342, 482] kind=swing-dependent
windows=297 S_holds=281 (0.946) median_spc=4.022 median_aspect=126.31
=== ctl_neg_RRLLLLRRL [2800000000, 3000000000] negative control (log growth)
ymax P=8: only 0 cycles in window, skipped
candidates tried (direction x P x p): 0
BEST NO_DATA
windows=0 S_holds=0 (0.000) median_spc=nan median_aspect=nan
=== RLLLLRRL [12401085717, 12601085717] census-escape window
ymax P=8: cycles=77 distinct_swing=77 best p=6 a=1059 exact=0.0909 b=[-31135118, -31378538, -30984198, -31342574, -31170082, -31319534]
candidates tried (direction x P x p): 192
BEST NO_EXACT_CLOCK: dir=xmax P=1 p=6 a=8 exact_fraction=0.1

## Post hoc: the registered phase-period cap was too small for RLLLLRRL
The registered test fixed p <= 6. For `RLLLLRRL` the residual `dt - 8 swing` is exactly periodic with period 19 once the run is past about 1.47e10 steps.

In [4]:
print((R / "stage1_posthoc_rlllrrl.txt").read_text()[-2600:])
print((R / "stage1_posthoc_maxp64.txt").read_text()[:1800])

= 17070097523 swing= 52011 share(80..110)=0.00 median=178 min=38
cycle 26400 t= 17238002751 swing= 52811 share(80..110)=0.00 median=178 min=38
cycle 26800 t= 17408467711 swing= 53611 share(80..110)=0.00 median=178 min=38
cycle 27200 t= 17581493131 swing= 54411 share(80..110)=0.00 median=178 min=38
cycle 27600 t= 17757078099 swing= 55211 share(80..110)=0.00 median=178 min=38
cycle 28000 t= 17935223719 swing= 56011 share(80..110)=0.00 median=178 min=38
cycle 28400 t= 18115928687 swing= 56811 share(80..110)=0.00 median=178 min=38
cycle 28800 t= 18299194219 swing= 57611 share(80..110)=0.00 median=178 min=38
cycle 29200 t= 18485019191 swing= 58411 share(80..110)=0.00 median=178 min=38
cycle 29600 t= 18673405023 swing= 59211 share(80..110)=0.00 median=178 min=38
cycle 30000 t= 18864350007 swing= 60011 share(80..110)=0.00 median=178 min=38
cycle 30400 t= 19057855799 swing= 60811 share(80..110)=0.00 median=178 min=38
cycle 30800 t= 19253922831 swing= 61611 share(80..110)=0.00 median=178 min=38

## `RLLLRLRLLL`: onset of the exact clock and board independence (G5)

In [5]:
print((R / "stage1_onset.txt").read_text())
print((R / "g5_compare.txt").read_text())

# commit 2ee0086; onset of the exact clock, derived patterns
xmin P=1 a=8 phase period 2: 32730 cycles; exact clock from cycle 2876 (of 32730); cycles in the exact regime 29854
onset: cycle 2876 ends t = 26351953955, swing 5809; the cycle before it ends t = 26351907669, swing 5807, residual -210
xmin P=1 a=8 phase period 19: 32766 cycles; exact clock from cycle 19596 (of 32766); cycles in the exact regime 13170
onset: cycle 19596 ends t = 14724680227, swing 39166; the cycle before it ends t = 14724364997, swing 39164, residual 46
xmax P=1 a=8 phase period 19: 32748 cycles; exact clock from cycle 19578 (of 32748); cycles in the exact regime 13170
onset: cycle 19578 ends t = 14724863567, swing 39167; the cycle before it ends t = 14724550185, swing 39165, residual 1910

# G5 comparison against the 32768-board run own exit step (the first attempt used the census exit step 28427043780, which is one revolution (262k steps) later because the census board check is one cell wider; that attempt 

## Stage 2 feasibility look (unregistered)
Trace over five revolutions of the square spiral of `RLLLRLRLLL` at swing about 6055, and the steady-state test of its outer band.

In [6]:
print((R / "spiral_probe_analysis.txt").read_text()[:2400])
print((R / "spiral_steady.txt").read_text()[:1400])

steps in trace 241446; bbox at T0 (x0 x1 y0 y1): [-3001, 2986, -3018, 3036], at T1: [-3006, 2991, -3023, 3041]; ant at T0 [-3001, 266, 3], at T1 [-3006, 271, 3]
depth (distance from the T0 bbox edge, negative = outside) range: -5 76
depth histogram (first 20 depths from the outside): [(-5, 3694), (-4, 9730), (-3, 12200), (-2, 12330), (-1, 12464), (0, 8934), (1, 3030), (2, 662), (3, 670), (4, 674), (5, 1174), (6, 1716), (7, 4574), (8, 7386), (9, 8460), (10, 9038), (11, 10718), (12, 10082), (13, 8118), (14, 8186)]
deepest depth visited: 76 ; steps at depth >= 8: 169594
velocity per 2000-step block, first 60 blocks (i, vx, vy):
[(0, 0, -1000), (2000, 0, -1000), (4000, 26, -974), (6000, 736, -264), (8000, 998, -2), (10000, 982, -18), (12000, 1000, 0), (14000, 974, -26), (16000, 892, 58), (18000, 380, 620), (20000, -4, 996), (22000, -8, 992), (24000, 0, 1000), (26000, -4, 996), (28000, 0, 1000), (30000, -640, 360), (32000, -994, 6), (34000, -992, 8), (36000, -982, 18), (38000, -1000, 0), (4

## What this shows, and what it does not
- Two of the three escapes (`RLLLRLRLLL`, `RLLLLRRL`) become square spirals with exact affine clocks after late onsets (2.635e10 and about 1.47e10 steps); the third is still a blob at 6e10. None is a column sweeper: the parent's window signature S fires in 0% of their windows.
- Registered predictions failed or were only partly right: G1 (p <= 6 cap), G2's onset range, G3, G4 (S does not track the clock); G5 and the positive control held; the registered negative control was vacuous.
- Not shown: that either spiral persists forever. The cell-level test says the outer band is a steady profile (99.2-99.7% of sampled cells equal after five revolutions) but the ant reads 76 cells deep and the path has about eight segment kinds, so the three-stage sweeper checker does not transfer and Stage 2 was not started.
- Next step if pursued: a registered pipeline with a periodic *ahead* background (generalising the rectangle cycler of the highway-decider experiment) and seam blocks.